# Site-aware sensitivity analysis for TRACK-FA composites

This notebook answers the site-effect follow-up without changing the existing model notebooks. It keeps the current `frda_only` and `control_aware` recipes, adds a third `site_aware` fold-local recipe, then evaluates all three with the same global Standardised Response Mean and patient-adaptive model runners.

**Validation contract.** Participant defines the outer split. The `site_aware` recipe is selected inside each outer-training fold only. Friedreich ataxia training data fit scaling and model coefficients. Held-out Friedreich ataxia and controls are scored only after the feature recipe, tuning, scaling, and model weights are locked.

## Short literature note

- **ComBat / neuroCombat** estimates site/scanner mean and variance effects in derived neuroimaging features while preserving specified biological covariates. It is widely used for multi-site magnetic resonance imaging, but can be risky when site and biology are strongly confounded (Fortin et al., 2018, NeuroImage, doi:10.1016/j.neuroimage.2017.11.024).
- **Longitudinal ComBat** extends this idea for repeated magnetic resonance imaging and is conceptually closer to TRACK-FA because the target is within-participant longitudinal change (Beer et al., 2020, NeuroImage, doi:10.1016/j.neuroimage.2020.117129).
- **Mixed-effects and participant-clustered inference** test whether site explains longitudinal score change without changing the biomarker. This is the safer first diagnostic when some participants contribute two annual pairs.
- **Practical rule for this project:** first report adjusted, participant-aware site diagnostics. If a concerning site effect remains, evaluate site-aware feature selection or fold-local harmonisation as sensitivity analyses. Any harmonisation must be fitted only inside the training fold to avoid train/test leakage.

## 1. Setup and artifact loading

The notebook reads the existing experiment from the authoritative project directory if this worktree does not yet contain the result artifacts, but all new outputs are written into this worktree.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "src").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not find repository root")


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import Config
from src.data.trackfa_pairs import trackfa_pairs_to_long
from src.eval.control_aware_selection import (
    control_aware_feature_ranking,
    single_feature_effect_table,
    wide_cohort_to_pair_long,
)
from src.eval.recipe_models import (
    DEFAULT_MODULATOR_SETS,
    comparison_table,
    run_adaptive_recipe_comparison,
    run_srm_recipe_comparison,
)
from src.features.panels import a_priori_70_feature_names
from src.reporting.experiment_artifacts import (
    FEATURE_RECIPE_COLUMNS,
    read_experiment_contract,
    read_table_artifact,
    split_participants_for_fold,
    validate_feature_recipe,
    validate_fold_manifest,
    validate_oof_uniqueness,
    write_table_artifact,
)

RUN_ID = "trackfa_70_feature_comparison_v1"
LOCAL_RUN_DIR = REPO_ROOT / "results" / "experiments" / RUN_ID
REFERENCE_RUN_DIR = Path("/Users/robertwang/Documents/New_project/biomarkers/results/experiments") / RUN_ID
RUN_DIR = LOCAL_RUN_DIR if (LOCAL_RUN_DIR / "manifest.json").exists() else REFERENCE_RUN_DIR
OUTPUT_DIR = LOCAL_RUN_DIR / "site_aware"
SELECTION_DIR = RUN_DIR / "selections"
MODEL_OUTPUT_DIR = OUTPUT_DIR / "models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_BOOT = 300
FEATURE_COUNT = 16
CONTROL_PENALTY = 0.5
SITE_PENALTY = 1.0
RANDOM_SEED = 42

manifest, folds = read_experiment_contract(RUN_DIR)
feature_names = a_priori_70_feature_names()
validate_fold_manifest(folds)

frda_recipe = read_table_artifact(
    SELECTION_DIR / "frda_only_features_by_fold.csv",
    schema="feature_recipe",
    manifest=manifest,
    panel=feature_names,
)
control_recipe = read_table_artifact(
    SELECTION_DIR / "control_aware_features_by_fold.csv",
    schema="feature_recipe",
    manifest=manifest,
    panel=feature_names,
)

pairs = pd.read_csv(manifest["data_path"], low_memory=False)
frda_long = trackfa_pairs_to_long(pairs)
wide_path = Path(manifest["data_path"]).with_name("trackfa_merged_wide.csv")
wide = pd.read_csv(wide_path, low_memory=False)
control_long = wide_cohort_to_pair_long(wide, feature_names, cohort_value=1)
control_meta = wide.loc[pd.to_numeric(wide["study_group"], errors="coerce").eq(1), ["ID", "age", "gender"]].copy()
control_meta["subject"] = control_meta["ID"].astype(str).str.replace(r"^TRACKFA_", "", regex=True)
control_meta = control_meta.rename(columns={"gender": "sex"})
for col in ["age", "sex"]:
    control_meta[col] = pd.to_numeric(control_meta[col], errors="coerce")
control_long = control_long.merge(control_meta[["subject", "age", "sex"]].drop_duplicates("subject"), on="subject", how="left")

print(f"Reading base experiment from: {RUN_DIR}")
print(f"Writing site-aware outputs to: {OUTPUT_DIR}")
display(pd.DataFrame([
    {"cohort": "FRDA", "participants": frda_long["subject"].nunique(), "annual_pairs": frda_long["pair_id"].nunique()},
    {"cohort": "Control", "participants": control_long["subject"].nunique(), "annual_pairs": control_long["pair_id"].nunique()},
    {"cohort": "Panel", "participants": np.nan, "annual_pairs": len(feature_names)},
]))

Reading base experiment from: /Users/robertwang/Documents/New_project/biomarkers/results/experiments/trackfa_70_feature_comparison_v1
Writing site-aware outputs to: /Users/robertwang/.codex/worktrees/790f/biomarkers/results/experiments/trackfa_70_feature_comparison_v1/site_aware


,cohort,participants,annual_pairs
0,FRDA,117.0,207
1,Control,95.0,190
2,Panel,NaN,70


## 2. Compact site-diagnostic helpers

The adjusted site screen uses out-of-fold visit scores, converts them to annual pair deltas, and tests whether site explains residual variation after covariates. The cluster-robust Wald test clusters by participant, while the bootstrap confidence interval resamples participants so two annual pairs from the same participant stay together.

In [2]:
from scipy import stats


def pair_deltas_from_scores(scores: pd.DataFrame) -> pd.DataFrame:
    validate_oof_uniqueness(scores)
    meta_cols = ["run_id", "model", "selection_strategy", "cohort", "outer_fold", "participant_id", "pair_id", "interval", "site", "feature_count", "modulator_recipe"]
    id_cols = ["run_id", "model", "selection_strategy", "cohort", "outer_fold", "participant_id", "pair_id"]
    meta = scores[meta_cols].drop_duplicates(id_cols)
    visit_scores = scores.pivot_table(index=id_cols, columns="visit", values="score", aggfunc="mean")
    out = visit_scores[[1, 2]].dropna().reset_index().rename(columns={1: "baseline_score", 2: "followup_score"})
    out["delta"] = out["followup_score"] - out["baseline_score"]
    return meta.merge(out[[*id_cols, "baseline_score", "followup_score", "delta"]], on=id_cols, how="inner")


def covariates_from_pairs_and_controls() -> pd.DataFrame:
    frda_cov = pairs[[c for c in ["patient_id", "site", "age", "gender", "disease_duration", "gaa_1", "mfars_total_baseline", "sara_total_baseline"] if c in pairs]].copy()
    frda_cov = frda_cov.rename(columns={"patient_id": "pair_id", "gender": "sex"})
    frda_cov["cohort"] = "FRDA"

    control_cov = control_long[[c for c in ["pair_id", "site", "age", "sex"] if c in control_long]].drop_duplicates("pair_id").copy()
    control_cov["cohort"] = "Control"
    for col in ["disease_duration", "gaa_1", "mfars_total_baseline", "sara_total_baseline"]:
        if col not in control_cov:
            control_cov[col] = np.nan
    common = ["pair_id", "cohort", "site", "age", "sex", "disease_duration", "gaa_1", "mfars_total_baseline", "sara_total_baseline"]
    cov = pd.concat([frda_cov.reindex(columns=common), control_cov.reindex(columns=common)], ignore_index=True)
    for col in common:
        if col not in ["pair_id", "cohort"]:
            cov[col] = pd.to_numeric(cov[col], errors="coerce")
    return cov.drop_duplicates(["cohort", "pair_id"])


PAIR_COVARIATES = covariates_from_pairs_and_controls()


def attach_covariates(deltas: pd.DataFrame) -> pd.DataFrame:
    out = deltas.merge(PAIR_COVARIATES, on=["cohort", "pair_id"], how="left", suffixes=("", "_cov"))
    if "site_cov" in out:
        out["site"] = out["site"].where(out["site"].notna(), out["site_cov"])
        out = out.drop(columns=["site_cov"])
    return out


def covariate_columns(df: pd.DataFrame, cohort: str) -> tuple[list[str], list[str]]:
    numeric = ["baseline_score"]
    for col in ["age", "disease_duration", "gaa_1", "mfars_total_baseline", "sara_total_baseline"]:
        if col in df and df[col].notna().sum() >= 20 and df[col].nunique(dropna=True) > 1:
            if cohort == "Control" and col in {"disease_duration", "gaa_1", "mfars_total_baseline", "sara_total_baseline"}:
                continue
            numeric.append(col)
    categorical = []
    for col in ["sex", "interval"]:
        if col in df and df[col].nunique(dropna=True) >= 2:
            categorical.append(col)
    return list(dict.fromkeys(numeric)), categorical


def design_matrix(df: pd.DataFrame, numeric: list[str], categorical: list[str]) -> tuple[np.ndarray, list[str]]:
    pieces = [pd.Series(1.0, index=df.index, name="Intercept")]
    names = ["Intercept"]
    for col in numeric:
        if col not in df:
            continue
        values = pd.to_numeric(df[col], errors="coerce")
        if values.nunique(dropna=True) <= 1:
            continue
        centered = values - values.mean()
        scale = values.std(ddof=0)
        if np.isfinite(scale) and scale > 0:
            centered = centered / scale
        pieces.append(centered.rename(col))
        names.append(col)
    for col in categorical:
        if col not in df:
            continue
        dummies = pd.get_dummies(df[col].astype("category"), prefix=col, drop_first=True, dtype=float)
        for dummy in dummies:
            pieces.append(dummies[dummy].rename(dummy))
            names.append(dummy)
    X = pd.concat(pieces, axis=1).to_numpy(dtype=float)
    return X, names


def ols_fit(y: np.ndarray, X: np.ndarray) -> dict[str, object]:
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ beta
    rss = float(np.sum(resid ** 2))
    rank = int(np.linalg.matrix_rank(X))
    return {"beta": beta, "resid": resid, "rss": rss, "rank": rank}


def cluster_covariance(X: np.ndarray, resid: np.ndarray, clusters: pd.Series) -> np.ndarray:
    xtx_inv = np.linalg.pinv(X.T @ X)
    meat = np.zeros((X.shape[1], X.shape[1]), dtype=float)
    cluster_values = clusters.astype(str).to_numpy()
    unique_clusters = np.unique(cluster_values)
    for value in unique_clusters:
        mask = cluster_values == value
        xu = X[mask].T @ resid[mask]
        meat += np.outer(xu, xu)
    n, p = X.shape
    g = len(unique_clusters)
    if g > 1 and n > p:
        meat *= (g / (g - 1)) * ((n - 1) / (n - p))
    return xtx_inv @ meat @ xtx_inv


def site_screen_one(df: pd.DataFrame, *, cohort: str, n_boot: int = 300, seed: int = 42) -> dict[str, object]:
    work = df.copy()
    required = ["delta", "baseline_score", "site", "participant_id"]
    if any(col not in work for col in required):
        return {"n_pairs": 0, "site_levels": 0, "site_partial_r2": np.nan, "site_f_p_value": np.nan, "site_cluster_p_value": np.nan}
    work = work.replace([np.inf, -np.inf], np.nan).dropna(subset=required)
    if len(work) < 20 or work["site"].nunique(dropna=True) < 2:
        return {"n_pairs": int(len(work)), "site_levels": int(work["site"].nunique(dropna=True)), "site_partial_r2": np.nan, "site_f_p_value": np.nan, "site_cluster_p_value": np.nan}
    for col in ["site", "sex", "interval"]:
        if col in work:
            work[col] = work[col].astype("category")
    numeric, categorical = covariate_columns(work, cohort)
    keep_cols = ["delta", "site", "participant_id", *numeric, *categorical]
    work = work.dropna(subset=[c for c in keep_cols if c in work]).copy()
    if len(work) < 20 or work["site"].nunique(dropna=True) < 2:
        return {"n_pairs": int(len(work)), "site_levels": int(work["site"].nunique(dropna=True)), "site_partial_r2": np.nan, "site_f_p_value": np.nan, "site_cluster_p_value": np.nan}

    y = pd.to_numeric(work["delta"], errors="coerce").to_numpy(dtype=float)
    X_reduced, reduced_names = design_matrix(work, numeric, categorical)
    X_full, full_names = design_matrix(work, numeric, [*categorical, "site"])
    reduced = ols_fit(y, X_reduced)
    full = ols_fit(y, X_full)
    rss_reduced = float(reduced["rss"])
    rss_full = float(full["rss"])
    rank_reduced = int(reduced["rank"])
    rank_full = int(full["rank"])
    df_num = max(rank_full - rank_reduced, 0)
    df_den = max(len(y) - rank_full, 0)
    partial_r2 = max(0.0, (rss_reduced - rss_full) / rss_reduced) if rss_reduced > 0 else np.nan
    if df_num > 0 and df_den > 0 and rss_full > 0:
        f_stat = ((rss_reduced - rss_full) / df_num) / (rss_full / df_den)
        f_p_value = float(stats.f.sf(f_stat, df_num, df_den))
    else:
        f_stat = np.nan
        f_p_value = np.nan

    site_terms = [i for i, name in enumerate(full_names) if name.startswith("site_")]
    if site_terms:
        cov = cluster_covariance(X_full, np.asarray(full["resid"], dtype=float), work["participant_id"])
        beta_site = np.asarray(full["beta"], dtype=float)[site_terms]
        cov_site = cov[np.ix_(site_terms, site_terms)]
        try:
            wald_stat = float(beta_site.T @ np.linalg.pinv(cov_site) @ beta_site)
            cluster_p = float(stats.chi2.sf(wald_stat, len(site_terms)))
        except Exception:
            wald_stat = np.nan
            cluster_p = np.nan
    else:
        wald_stat = np.nan
        cluster_p = np.nan

    rng = np.random.default_rng(seed)
    participant_ids = work["participant_id"].astype(str).drop_duplicates().to_numpy()
    boot_r2 = []
    for _ in range(int(n_boot)):
        sampled = rng.choice(participant_ids, size=len(participant_ids), replace=True)
        boot = pd.concat([work.loc[work["participant_id"].astype(str).eq(pid)] for pid in sampled], ignore_index=True)
        if boot["site"].nunique(dropna=True) < 2:
            continue
        try:
            b_y = pd.to_numeric(boot["delta"], errors="coerce").to_numpy(dtype=float)
            b_reduced = ols_fit(b_y, design_matrix(boot, numeric, categorical)[0])
            b_full = ols_fit(b_y, design_matrix(boot, numeric, [*categorical, "site"])[0])
            if float(b_reduced["rss"]) > 0:
                boot_r2.append(max(0.0, (float(b_reduced["rss"]) - float(b_full["rss"])) / float(b_reduced["rss"])))
        except Exception:
            continue
    return {
        "n_pairs": int(len(work)),
        "n_participants": int(work["participant_id"].nunique()),
        "site_levels": int(work["site"].nunique(dropna=True)),
        "covariates": ", ".join([*numeric, *categorical]),
        "site_partial_r2": partial_r2,
        "site_f_stat": float(f_stat) if np.isfinite(f_stat) else np.nan,
        "site_f_p_value": float(f_p_value) if np.isfinite(f_p_value) else np.nan,
        "site_cluster_wald": wald_stat,
        "site_cluster_p_value": cluster_p,
        "site_partial_r2_boot_low": float(np.quantile(boot_r2, 0.025)) if boot_r2 else np.nan,
        "site_partial_r2_boot_high": float(np.quantile(boot_r2, 0.975)) if boot_r2 else np.nan,
    }


def adjusted_site_screen_from_oof(oof: pd.DataFrame, *, n_boot: int = 300, seed: int = 42) -> tuple[pd.DataFrame, pd.DataFrame]:
    deltas = attach_covariates(pair_deltas_from_scores(oof))
    rows = []
    for keys, group in deltas.groupby(["model", "selection_strategy", "cohort"], sort=False):
        model, strategy, cohort = keys
        row = {"model": model, "selection_strategy": strategy, "cohort": cohort}
        row.update(site_screen_one(group, cohort=str(cohort), n_boot=n_boot, seed=seed))
        rows.append(row)
    return deltas, pd.DataFrame(rows)


def site_stratified_summary(deltas: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for keys, group in deltas.groupby(["model", "selection_strategy", "cohort", "site"], dropna=False, sort=False):
        model, strategy, cohort, site = keys
        values = pd.to_numeric(group["delta"], errors="coerce").dropna()
        sd = values.std(ddof=1)
        rows.append({
            "model": model,
            "selection_strategy": strategy,
            "cohort": cohort,
            "site": site,
            "n_pairs": int(len(values)),
            "n_participants": int(group["participant_id"].nunique()),
            "mean_delta": float(values.mean()) if len(values) else np.nan,
            "sd_delta": float(sd) if len(values) > 1 else np.nan,
            "d_z": float(values.mean() / sd) if len(values) > 1 and sd != 0 else np.nan,
        })
    return pd.DataFrame(rows)


## 3. Existing-model adjusted site diagnostics

These tables diagnose the already-run out-of-fold composite scores before any new site-aware feature selection is introduced.

In [3]:
existing_oof_parts = []
for model_dir in [RUN_DIR / "models" / "srm_global_linear", RUN_DIR / "models" / "patient_adaptive"]:
    oof_path = model_dir / "oof_visit_scores.csv"
    if oof_path.exists():
        existing_oof_parts.append(read_table_artifact(oof_path, schema="oof_visit_scores", manifest=manifest))
existing_oof = pd.concat(existing_oof_parts, ignore_index=True)
existing_deltas, existing_site_screen = adjusted_site_screen_from_oof(existing_oof, n_boot=N_BOOT, seed=RANDOM_SEED)
existing_site_summary = site_stratified_summary(existing_deltas)

existing_deltas.to_csv(OUTPUT_DIR / "existing_score_deltas_with_covariates.csv", index=False)
existing_site_screen.to_csv(OUTPUT_DIR / "existing_site_adjusted_screen.csv", index=False)
existing_site_summary.to_csv(OUTPUT_DIR / "existing_site_stratified_delta_summary.csv", index=False)

print("Adjusted site screen for existing model outputs")
display(existing_site_screen.round(4))
print("Site-stratified score-change summary for existing model outputs")
display(existing_site_summary.round(4).head(24))

Adjusted site screen for existing model outputs


,model,selection_strategy,cohort,n_pairs,n_participants,site_levels,covariates,site_partial_r2,site_f_stat,site_f_p_value,site_cluster_wald,site_cluster_p_value,site_partial_r2_boot_low,site_partial_r2_boot_high
0,srm_global_linear,frda_only,FRDA,203,115,6,"baseline_score, age, disease_duration, gaa_1, ...",0.0700,2.8444,0.0168,35.5291,0.0000,0.0356,0.1551
1,srm_global_linear,frda_only,Control,126,67,6,"baseline_score, age, sex, interval",0.0528,1.2809,0.2769,20.0473,0.0012,0.0294,0.1359
2,srm_global_linear,control_aware,FRDA,203,115,6,"baseline_score, age, disease_duration, gaa_1, ...",0.0698,2.8369,0.0170,31.9139,0.0000,0.0324,0.1513
3,srm_global_linear,control_aware,Control,126,67,6,"baseline_score, age, sex, interval",0.0488,1.1790,0.3238,24.1176,0.0002,0.0262,0.1152
4,patient_adaptive_strategy_specific,frda_only,FRDA,203,115,6,"baseline_score, age, disease_duration, gaa_1, ...",0.0611,2.4602,0.0346,17.5856,0.0035,0.0301,0.1481
5,patient_adaptive_strategy_specific,frda_only,Control,126,67,6,"baseline_score, age, sex, interval",0.0311,0.7385,0.5961,17.3660,0.0039,0.0165,0.0856
6,patient_adaptive_strategy_specific,control_aware,FRDA,203,115,6,"baseline_score, age, disease_duration, gaa_1, ...",0.0547,2.1862,0.0574,15.5957,0.0081,0.0238,0.1310
7,patient_adaptive_strategy_specific,control_aware,Control,126,67,6,"baseline_score, age, sex, interval",0.0217,0.5102,0.7681,10.1406,0.0713,0.0095,0.0729
8,patient_adaptive_common_modulator,frda_only,FRDA,203,115,6,"baseline_score, age, disease_duration, gaa_1, ...",0.0645,2.6069,0.0263,25.4757,0.0001,0.0287,0.1497
9,patient_adaptive_common_modulator,frda_only,Control,126,67,6,"baseline_score, age, sex, interval",0.0266,0.6293,0.6778,16.6805,0.0051,0.0135,0.0861


Site-stratified score-change summary for existing model outputs


,model,selection_strategy,cohort,site,n_pairs,n_participants,mean_delta,sd_delta,d_z
0,srm_global_linear,frda_only,FRDA,2.0,46,25,0.9042,0.8990,1.0057
1,srm_global_linear,frda_only,FRDA,5.0,68,38,0.8371,1.2596,0.6646
2,srm_global_linear,frda_only,FRDA,7.0,13,7,0.1172,0.7625,0.1538
3,srm_global_linear,frda_only,FRDA,1.0,33,19,1.5143,1.2423,1.2189
4,srm_global_linear,frda_only,FRDA,6.0,33,20,0.7095,1.3371,0.5306
5,srm_global_linear,frda_only,FRDA,3.0,14,8,0.8069,1.0573,0.7631
6,srm_global_linear,frda_only,Control,2.0,31,16,0.1041,0.8390,0.1241
7,srm_global_linear,frda_only,Control,5.0,28,15,-0.2254,1.0914,-0.2065
8,srm_global_linear,frda_only,Control,7.0,12,6,0.4160,0.8276,0.5027
9,srm_global_linear,frda_only,Control,6.0,21,12,-0.0945,0.8261,-0.1144


## 4. Fold-local site-aware feature recipe

The new selection score starts from the supervisor-approved control-aware idea and adds a site penalty computed on Friedreich ataxia training feature deltas only:

`site_aware_score = FRDA interval-consistency - 0.5 * same-direction control effect - 1.0 * adjusted site partial R2`

This is intentionally conservative: controls penalise non-specific progression, while the site term discourages features whose longitudinal changes remain strongly site-associated in the Friedreich ataxia training fold.

In [4]:
def training_subset(frame: pd.DataFrame, participant_ids: set[str]) -> pd.DataFrame:
    return frame.loc[frame["subject"].astype(str).isin(participant_ids)].copy()


def feature_delta_table(frame: pd.DataFrame, feature: str) -> pd.DataFrame:
    meta_cols = [c for c in ["pair_id", "subject", "interval", "site", "age", "sex", "disease_duration", "gaa_1"] if c in frame]
    meta = frame[meta_cols].drop_duplicates("pair_id")
    wide_feature = frame.pivot_table(index="pair_id", columns="visit", values=feature, aggfunc="mean")
    if 1 not in wide_feature or 2 not in wide_feature:
        return pd.DataFrame(columns=["pair_id", "subject", "interval", "site", "delta"])
    out = wide_feature[[1, 2]].dropna().reset_index()
    out["delta"] = out[2] - out[1]
    out = meta.merge(out[["pair_id", "delta"]], on="pair_id", how="inner")
    return out.rename(columns={"subject": "participant_id"})


def feature_site_partial_r2(frda_train: pd.DataFrame, feature: str) -> float:
    deltas = feature_delta_table(frda_train, feature)
    if deltas.empty:
        return np.nan
    deltas["baseline_score"] = 0.0
    try:
        result = site_screen_one(deltas, cohort="FRDA", n_boot=0, seed=RANDOM_SEED)
        return float(result.get("site_partial_r2", np.nan))
    except Exception:
        return np.nan


def site_aware_feature_ranking(frda_train: pd.DataFrame, control_train: pd.DataFrame) -> pd.DataFrame:
    ranking = control_aware_feature_ranking(
        frda_train,
        control_train,
        feature_names,
        control_penalty=CONTROL_PENALTY,
        pair_col="pair_id",
        participant_col="subject",
        visit_col="visit",
    ).copy()
    ranking["site_partial_r2"] = [feature_site_partial_r2(frda_train, feature) for feature in ranking["feature"]]
    ranking["site_penalty"] = ranking["site_partial_r2"].fillna(0.0).clip(lower=0.0) * SITE_PENALTY
    ranking["site_aware_score"] = ranking["frda_consistency"] - CONTROL_PENALTY * ranking["same_direction_control_penalty"] - ranking["site_penalty"]
    ranking = ranking.sort_values(
        ["site_aware_score", "frda_consistency", "same_direction_control_penalty", "site_partial_r2", "feature"],
        ascending=[False, False, True, True, True],
        kind="mergesort",
    ).reset_index(drop=True)
    ranking["rank"] = np.arange(1, len(ranking) + 1)
    return ranking


ranking_parts = []
recipe_rows = []
for outer_fold in sorted(pd.to_numeric(folds.loc[folds["cohort"].eq("FRDA"), "outer_fold"]).astype(int).unique()):
    frda_train_ids, _ = split_participants_for_fold(folds, cohort="FRDA", outer_fold=int(outer_fold))
    control_train_ids, _ = split_participants_for_fold(folds, cohort="Control", outer_fold=int(outer_fold))
    frda_train = training_subset(frda_long, frda_train_ids)
    control_train = training_subset(control_long, control_train_ids)
    ranking = site_aware_feature_ranking(frda_train, control_train)
    ranking.insert(0, "outer_fold", int(outer_fold))
    ranking_parts.append(ranking)
    selected = set(ranking.head(FEATURE_COUNT)["feature"].astype(str))
    for _, row in ranking.iterrows():
        recipe_rows.append({
            "strategy": "site_aware",
            "outer_fold": int(outer_fold),
            "feature": str(row["feature"]),
            "rank": int(row["rank"]),
            "selected": str(row["feature"]) in selected,
            "selection_score": float(row["site_aware_score"]),
            "frda_pooled_d_z": row.get("frda_pooled_d_z", np.nan),
            "frda_v1_v2_d_z": row.get("frda_v1_v2_d_z", np.nan),
            "frda_v2_v3_d_z": row.get("frda_v2_v3_d_z", np.nan),
            "control_pooled_d_z": row.get("control_pooled_d_z", np.nan),
            "control_v1_v2_d_z": row.get("control_v1_v2_d_z", np.nan),
            "control_v2_v3_d_z": row.get("control_v2_v3_d_z", np.nan),
            "train_frda_participants": int(frda_train["subject"].nunique()),
            "train_frda_pairs": int(frda_train["pair_id"].nunique()),
            "train_control_participants": int(control_train["subject"].nunique()),
            "train_control_pairs": int(control_train["pair_id"].nunique()),
        })

site_ranking = pd.concat(ranking_parts, ignore_index=True)
site_recipe = pd.DataFrame(recipe_rows, columns=FEATURE_RECIPE_COLUMNS)
validate_feature_recipe(site_recipe, panel=feature_names)

site_ranking.to_csv(OUTPUT_DIR / "site_aware_feature_ranking_by_fold.csv", index=False)
write_table_artifact(
    OUTPUT_DIR / "site_aware_features_by_fold.csv",
    site_recipe,
    schema="feature_recipe",
    manifest=manifest,
    panel=feature_names,
    metadata={"control_penalty": CONTROL_PENALTY, "site_penalty": SITE_PENALTY, "feature_count": FEATURE_COUNT},
)

selected_display = site_recipe.loc[site_recipe["selected"]].groupby("outer_fold")["feature"].apply(lambda x: ", ".join(x)).reset_index()
print("Selected site-aware features by fold")
display(selected_display)
print("Top site-aware ranking rows")
display(site_ranking.head(20).round(4))

Selected site-aware features by fold


,outer_fold,feature
0,1,"Midbrain, Pons, Cerebellum_Cortex_CerebNet, Ce..."
1,2,"Cerebellum_Cortex_CerebNet, Midbrain, Pons, Ce..."
2,3,"Cerebellum_Cortex_CerebNet, Pons, Cerebellum_W..."
3,4,"Cerebellum_Cortex_CerebNet, Cerebellum_WM_Cere..."
4,5,"Cerebellum_WM_CerebNet, Cerebellum_Cortex_Cere..."


Top site-aware ranking rows


,outer_fold,rank,feature,frda_pooled_d_z,frda_v1_v2_d_z,frda_v2_v3_d_z,frda_direction,frda_consistency,control_pooled_d_z,control_v1_v2_d_z,control_v2_v3_d_z,same_direction_control_penalty,control_aware_score,eligible,frda_pooled_n,control_pooled_n,site_partial_r2,site_penalty,site_aware_score
0,1,1,Midbrain,-0.4117,-0.4054,-0.4178,-1.0,0.4054,0.3523,0.3348,0.3673,0.0000,0.4054,True,165,123,0.0419,0.0419,0.3636
1,1,2,Pons,-0.6041,-0.8423,-0.3998,-1.0,0.3998,0.5117,0.3725,0.6827,0.0000,0.3998,True,165,123,0.0788,0.0788,0.3210
2,1,3,Cerebellum_Cortex_CerebNet,-0.6478,-0.9923,-0.3811,-1.0,0.3811,-0.0204,-0.2062,0.2167,0.0204,0.3709,True,165,123,0.0614,0.0614,0.3095
3,1,4,Cerebellum_WM_CerebNet,-0.4488,-0.5920,-0.3195,-1.0,0.3195,0.2792,0.2436,0.3187,0.0000,0.3195,True,165,123,0.0505,0.0505,0.2690
4,1,5,FA_PTR,-0.2868,-0.2658,-0.3073,-1.0,0.2658,0.0696,0.2809,-0.1314,0.0000,0.2658,True,165,104,0.0177,0.0177,0.2481
5,1,6,Thalamus,-0.4026,-0.5590,-0.2470,-1.0,0.2470,0.0681,-0.0055,0.1460,0.0000,0.2470,True,165,123,0.0580,0.0580,0.1889
6,1,7,FA_sCC,-0.2183,-0.1990,-0.2357,-1.0,0.1990,0.0780,0.4085,-0.2447,0.0000,0.1990,True,165,104,0.0459,0.0459,0.1531
7,1,8,Lateral_Ventricle,0.4042,0.5834,0.3021,1.0,0.3021,0.2598,0.2649,0.2592,0.2598,0.1722,True,165,123,0.0192,0.0192,0.1530
8,1,9,FA_ILF_IFOF,-0.2177,-0.2782,-0.1628,-1.0,0.1628,0.1322,0.1168,0.2228,0.0000,0.1628,True,165,104,0.0168,0.0168,0.1459
9,1,10,RD_PTR,0.1894,0.2234,0.1525,1.0,0.1525,-0.1412,-0.1826,-0.1036,0.0000,0.1525,True,165,104,0.0243,0.0243,0.1282


## 5. Global Standardised Response Mean with three recipes

This reruns the same global model on `frda_only`, `control_aware`, and `site_aware` features. Bootstrapping is set to 300 for runtime efficiency.

In [5]:
combined_recipe = pd.concat([frda_recipe, control_recipe, site_recipe], ignore_index=True)
validate_feature_recipe(combined_recipe, panel=feature_names)

srm_result = run_srm_recipe_comparison(
    frda_long,
    control_long,
    folds,
    combined_recipe,
    run_id=RUN_ID,
    inner_folds=3,
    seed=int(manifest["seed"]),
    n_boot=N_BOOT,
)

srm_dir = MODEL_OUTPUT_DIR / "srm_global_linear_site_aware_comparison"
srm_dir.mkdir(parents=True, exist_ok=True)
write_table_artifact(srm_dir / "oof_visit_scores.csv", srm_result["oof_visit_scores"], schema="oof_visit_scores", manifest=manifest)
write_table_artifact(srm_dir / "performance.csv", srm_result["performance"], schema="performance", manifest=manifest)
write_table_artifact(srm_dir / "coefficients.csv", srm_result["coefficients"], schema="coefficients", manifest=manifest)
srm_result["fold_parameters"].to_csv(srm_dir / "fold_parameters.csv", index=False)
srm_result["tuning"].to_csv(srm_dir / "inner_tuning.csv", index=False)
srm_result["site_diagnostics"].to_csv(srm_dir / "unadjusted_site_diagnostics.csv", index=False)
srm_result["comparison"].to_csv(srm_dir / "headline_comparison.csv", index=False)

srm_deltas, srm_adjusted_site = adjusted_site_screen_from_oof(srm_result["oof_visit_scores"], n_boot=N_BOOT, seed=RANDOM_SEED)
srm_site_summary = site_stratified_summary(srm_deltas)
srm_deltas.to_csv(srm_dir / "score_deltas_with_covariates.csv", index=False)
srm_adjusted_site.to_csv(srm_dir / "site_adjusted_screen.csv", index=False)
srm_site_summary.to_csv(srm_dir / "site_stratified_delta_summary.csv", index=False)

srm_display = srm_result["comparison"].copy()
print("Global SRM performance: original, control-aware, and site-aware feature selections")
display(srm_display.round(3))
print("Global SRM adjusted site diagnostics")
display(srm_adjusted_site.round(4))

Global SRM performance: original, control-aware, and site-aware feature selections


,model,selection_strategy,frda_v1_v2_ci_high,frda_v2_v3_ci_high,frda_pooled_annual_ci_high,frda_v1_v2_ci_low,frda_v2_v3_ci_low,frda_pooled_annual_ci_low,frda_v1_v2_d_z,frda_v2_v3_d_z,...,control_pooled_annual_n_pairs,control_v1_v2_n_participants,control_v2_v3_n_participants,control_pooled_annual_n_participants,control_v1_v2_p_delta_gt_0,control_v2_v3_p_delta_gt_0,control_pooled_annual_p_delta_gt_0,signed_frda_control_contrast,absolute_control_d_z,frda_interval_gap
0,srm_global_linear,control_aware,1.212,0.803,0.910,0.715,0.411,0.614,0.944,0.579,...,126,63,63,67,0.460,0.508,0.484,0.760,0.011,0.365
1,srm_global_linear,frda_only,1.249,0.783,0.924,0.738,0.408,0.614,0.959,0.569,...,126,63,63,67,0.508,0.540,0.524,0.715,0.033,0.390
2,srm_global_linear,site_aware,1.229,0.804,0.925,0.733,0.432,0.629,0.957,0.589,...,126,63,63,67,0.476,0.508,0.492,0.770,0.011,0.367


Global SRM adjusted site diagnostics


,model,selection_strategy,cohort,n_pairs,n_participants,site_levels,covariates,site_partial_r2,site_f_stat,site_f_p_value,site_cluster_wald,site_cluster_p_value,site_partial_r2_boot_low,site_partial_r2_boot_high
0,srm_global_linear,frda_only,FRDA,203,115,6,"baseline_score, age, disease_duration, gaa_1, ...",0.0700,2.8444,0.0168,35.5291,0.0000,0.0356,0.1551
1,srm_global_linear,frda_only,Control,126,67,6,"baseline_score, age, sex, interval",0.0528,1.2809,0.2769,20.0473,0.0012,0.0294,0.1359
2,srm_global_linear,control_aware,FRDA,203,115,6,"baseline_score, age, disease_duration, gaa_1, ...",0.0698,2.8369,0.0170,31.9139,0.0000,0.0324,0.1513
3,srm_global_linear,control_aware,Control,126,67,6,"baseline_score, age, sex, interval",0.0488,1.1790,0.3238,24.1176,0.0002,0.0262,0.1152
4,srm_global_linear,site_aware,FRDA,203,115,6,"baseline_score, age, disease_duration, gaa_1, ...",0.0693,2.8141,0.0178,35.4771,0.0000,0.0340,0.1538
5,srm_global_linear,site_aware,Control,126,67,6,"baseline_score, age, sex, interval",0.0579,1.4132,0.2247,22.0937,0.0005,0.0302,0.1414


## 6. Patient-adaptive sensitivity model with three recipes

The adaptive model is retained as a sensitivity analysis. Modulator choice remains nested inside the Friedreich ataxia training folds.

In [6]:
adaptive_config = Config(
    random_state=int(manifest["seed"]),
    interaction_en_alpha_grid=(0.03, 0.1, 0.3, 1.0, 3.0),
    interaction_en_l1_ratio_grid=(0.0,),
    interaction_inner_cv_splits=3,
    interaction_z_clip=2.75,
    interaction_tune_inner_cv=True,
)
adaptive_result = run_adaptive_recipe_comparison(
    frda_long,
    control_long,
    folds,
    combined_recipe,
    run_id=RUN_ID,
    modulator_sets=DEFAULT_MODULATOR_SETS,
    inner_folds=3,
    seed=int(manifest["seed"]),
    n_boot=N_BOOT,
    config=adaptive_config,
)

adaptive_dir = MODEL_OUTPUT_DIR / "patient_adaptive_site_aware_comparison"
adaptive_dir.mkdir(parents=True, exist_ok=True)
write_table_artifact(adaptive_dir / "oof_visit_scores.csv", adaptive_result["oof_visit_scores"], schema="oof_visit_scores", manifest=manifest)
write_table_artifact(adaptive_dir / "performance.csv", adaptive_result["performance"], schema="performance", manifest=manifest)
write_table_artifact(adaptive_dir / "coefficients.csv", adaptive_result["coefficients"], schema="coefficients", manifest=manifest)
adaptive_result["modulator_inner_scores"].to_csv(adaptive_dir / "modulator_inner_scores.csv", index=False)
adaptive_result["modulator_choices"].to_csv(adaptive_dir / "modulator_choices.csv", index=False)
adaptive_result["modulator_frequency"].to_csv(adaptive_dir / "modulator_frequency.csv", index=False)
adaptive_result["fold_parameters"].to_csv(adaptive_dir / "fold_parameters.csv", index=False)
adaptive_result["site_diagnostics"].to_csv(adaptive_dir / "unadjusted_site_diagnostics.csv", index=False)
adaptive_result["comparison"].to_csv(adaptive_dir / "headline_comparison.csv", index=False)

adaptive_deltas, adaptive_adjusted_site = adjusted_site_screen_from_oof(adaptive_result["oof_visit_scores"], n_boot=N_BOOT, seed=RANDOM_SEED)
adaptive_site_summary = site_stratified_summary(adaptive_deltas)
adaptive_deltas.to_csv(adaptive_dir / "score_deltas_with_covariates.csv", index=False)
adaptive_adjusted_site.to_csv(adaptive_dir / "site_adjusted_screen.csv", index=False)
adaptive_site_summary.to_csv(adaptive_dir / "site_stratified_delta_summary.csv", index=False)

print("Patient-adaptive performance: original, control-aware, and site-aware feature selections")
display(adaptive_result["comparison"].round(3))
print("Patient-adaptive adjusted site diagnostics")
display(adaptive_adjusted_site.round(4))
print("Adaptive modulator frequency")
display(adaptive_result["modulator_frequency"].round(3))

Patient-adaptive performance: original, control-aware, and site-aware feature selections


,model,selection_strategy,frda_v1_v2_ci_high,frda_v2_v3_ci_high,frda_pooled_annual_ci_high,frda_v1_v2_ci_low,frda_v2_v3_ci_low,frda_pooled_annual_ci_low,frda_v1_v2_d_z,frda_v2_v3_d_z,...,control_pooled_annual_n_pairs,control_v1_v2_n_participants,control_v2_v3_n_participants,control_pooled_annual_n_participants,control_v1_v2_p_delta_gt_0,control_v2_v3_p_delta_gt_0,control_pooled_annual_p_delta_gt_0,signed_frda_control_contrast,absolute_control_d_z,frda_interval_gap
0,patient_adaptive_common_modulator,control_aware,1.082,0.704,0.817,0.553,0.332,0.478,0.809,0.496,...,126,63,63,67,0.571,0.524,0.548,0.571,0.073,0.313
1,patient_adaptive_common_modulator,frda_only,1.090,0.800,0.865,0.600,0.418,0.564,0.825,0.598,...,126,63,63,67,0.556,0.556,0.556,0.624,0.080,0.227
2,patient_adaptive_common_modulator,site_aware,1.047,0.749,0.818,0.548,0.353,0.496,0.784,0.524,...,126,63,63,67,0.603,0.460,0.532,0.572,0.076,0.260
3,patient_adaptive_strategy_specific,control_aware,1.068,0.746,0.833,0.536,0.378,0.495,0.793,0.537,...,126,63,63,67,0.556,0.492,0.524,0.586,0.076,0.256
4,patient_adaptive_strategy_specific,frda_only,0.996,0.775,0.826,0.525,0.416,0.507,0.728,0.577,...,126,63,63,67,0.571,0.571,0.571,0.579,0.076,0.150
5,patient_adaptive_strategy_specific,site_aware,1.087,0.772,0.828,0.551,0.360,0.497,0.788,0.538,...,126,63,63,67,0.571,0.460,0.516,0.605,0.055,0.250


Patient-adaptive adjusted site diagnostics


,model,selection_strategy,cohort,n_pairs,n_participants,site_levels,covariates,site_partial_r2,site_f_stat,site_f_p_value,site_cluster_wald,site_cluster_p_value,site_partial_r2_boot_low,site_partial_r2_boot_high
0,patient_adaptive_strategy_specific,frda_only,FRDA,203,115,6,"baseline_score, age, disease_duration, gaa_1, ...",0.0611,2.4602,0.0346,17.5856,0.0035,0.0301,0.1481
1,patient_adaptive_strategy_specific,frda_only,Control,126,67,6,"baseline_score, age, sex, interval",0.0311,0.7385,0.5961,17.3660,0.0039,0.0165,0.0856
2,patient_adaptive_strategy_specific,control_aware,FRDA,203,115,6,"baseline_score, age, disease_duration, gaa_1, ...",0.0547,2.1862,0.0574,15.5957,0.0081,0.0238,0.1310
3,patient_adaptive_strategy_specific,control_aware,Control,126,67,6,"baseline_score, age, sex, interval",0.0217,0.5102,0.7681,10.1406,0.0713,0.0095,0.0729
4,patient_adaptive_strategy_specific,site_aware,FRDA,203,115,6,"baseline_score, age, disease_duration, gaa_1, ...",0.0471,1.8696,0.1015,13.6477,0.0180,0.0194,0.1238
5,patient_adaptive_strategy_specific,site_aware,Control,126,67,6,"baseline_score, age, sex, interval",0.0416,0.9978,0.4224,23.7646,0.0002,0.0225,0.1030
6,patient_adaptive_common_modulator,frda_only,FRDA,203,115,6,"baseline_score, age, disease_duration, gaa_1, ...",0.0740,3.0196,0.0120,22.5606,0.0004,0.0347,0.1648
7,patient_adaptive_common_modulator,frda_only,Control,126,67,6,"baseline_score, age, sex, interval",0.0361,0.8618,0.5091,21.5546,0.0006,0.0183,0.1009
8,patient_adaptive_common_modulator,control_aware,FRDA,203,115,6,"baseline_score, age, disease_duration, gaa_1, ...",0.0573,2.2971,0.0469,14.8562,0.0110,0.0213,0.1410
9,patient_adaptive_common_modulator,control_aware,Control,126,67,6,"baseline_score, age, sex, interval",0.0236,0.5554,0.7339,10.9638,0.0521,0.0110,0.0751


Adaptive modulator frequency


,comparison_mode,selection_strategy,modulators,folds_selected,selection_frequency
0,common_modulator,control_aware,disease_duration,3,0.6
1,common_modulator,control_aware,gaa_1,2,0.4
2,common_modulator,frda_only,disease_duration,3,0.6
3,common_modulator,frda_only,gaa_1,2,0.4
4,common_modulator,site_aware,disease_duration,3,0.6
5,common_modulator,site_aware,gaa_1,2,0.4
6,strategy_specific,control_aware,disease_duration,2,0.4
7,strategy_specific,control_aware,"disease_duration,gaa_1",1,0.2
8,strategy_specific,control_aware,gaa_1,2,0.4
9,strategy_specific,frda_only,disease_duration,1,0.2


## 7. Coefficients versus marginal single-feature effects

This table compares the global SRM site-aware coefficients with marginal single-feature longitudinal effects. Coefficients are conditional multivariable weights; the single-feature `d_z` columns describe each feature alone.

In [7]:
selected_site_features = sorted(site_recipe.loc[site_recipe["selected"], "feature"].unique())
frda_marginal = single_feature_effect_table(frda_long, selected_site_features, cohort="FRDA").rename(columns={
    "pooled_d_z": "frda_single_pooled_d_z",
    "v1_v2_d_z": "frda_single_v1_v2_d_z",
    "v2_v3_d_z": "frda_single_v2_v3_d_z",
})
control_marginal = single_feature_effect_table(control_long, selected_site_features, cohort="Control").rename(columns={
    "pooled_d_z": "control_single_pooled_d_z",
    "v1_v2_d_z": "control_single_v1_v2_d_z",
    "v2_v3_d_z": "control_single_v2_v3_d_z",
})
coef_summary = (
    srm_result["coefficients"]
    .loc[srm_result["coefficients"]["selection_strategy"].eq("site_aware")]
    .groupby("feature", as_index=False)
    .agg(mean_coefficient=("coefficient", "mean"), mean_abs_coefficient=("coefficient", lambda x: np.mean(np.abs(x))), folds=("outer_fold", "nunique"))
)
feature_frequency = (
    site_recipe.loc[site_recipe["selected"]]
    .groupby("feature", as_index=False)
    .size()
    .rename(columns={"size": "selected_folds"})
)
coef_effect_table = (
    coef_summary
    .merge(feature_frequency, on="feature", how="left")
    .merge(frda_marginal[["feature", "frda_single_pooled_d_z", "frda_single_v1_v2_d_z", "frda_single_v2_v3_d_z"]], on="feature", how="left")
    .merge(control_marginal[["feature", "control_single_pooled_d_z", "control_single_v1_v2_d_z", "control_single_v2_v3_d_z"]], on="feature", how="left")
    .sort_values(["mean_abs_coefficient", "selected_folds", "feature"], ascending=[False, False, True], kind="mergesort")
)
coef_effect_table.to_csv(OUTPUT_DIR / "site_aware_coefficient_marginal_effect_comparison.csv", index=False)
print("Site-aware SRM coefficients versus marginal feature effects")
display(coef_effect_table.round(4))

Site-aware SRM coefficients versus marginal feature effects


,feature,mean_coefficient,mean_abs_coefficient,folds,selected_folds,frda_single_pooled_d_z,frda_single_v1_v2_d_z,frda_single_v2_v3_d_z,control_single_pooled_d_z,control_single_v1_v2_d_z,control_single_v2_v3_d_z
10,Lateral_Ventricle,2.5174,2.5174,5,5,0.3853,0.5359,0.2861,0.2488,0.2831,0.2247
0,Cerebellum_Cortex_CerebNet,-2.2004,2.2004,5,5,-0.6204,-0.8780,-0.4031,-0.0193,-0.1688,0.1594
1,Cerebellum_WM_CerebNet,-1.5671,1.5671,5,5,-0.4251,-0.5100,-0.3424,0.2283,0.3189,0.1423
13,Pons,-1.4695,1.4695,5,5,-0.5517,-0.7772,-0.3591,0.5087,0.4107,0.6191
12,Midbrain,-1.2388,1.2388,5,5,-0.3521,-0.3877,-0.3238,0.3271,0.3031,0.3491
22,TotalBrainGMVol_nocereb,-1.1513,1.1513,5,5,-0.4128,-0.4022,-0.4228,-0.4899,-0.3185,-0.6859
21,Thalamus,-0.9244,0.9244,5,5,-0.3671,-0.4694,-0.2482,0.0905,0.0289,0.1550
6,FA_PTR,-0.6017,0.6017,5,5,-0.2529,-0.2071,-0.3037,0.0756,0.2558,-0.1254
16,RD_SCP,0.5762,0.5762,4,4,0.2601,0.1621,0.3543,-0.0850,-0.2731,0.0821
18,RD_bCC,-0.5308,0.5308,1,1,0.0953,0.0942,0.0960,-0.2240,-0.2464,-0.2030


## 8. Final comparison and acceptance summary

The current primary goal is not to remove every site difference at any cost. A useful site-aware model should keep strong Friedreich ataxia longitudinal sensitivity, keep controls near zero or opposite direction, and reduce adjusted site association.

In [8]:
def compact_acceptance(model_name: str, comparison: pd.DataFrame, site_screen: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in comparison.iterrows():
        strategy = row["selection_strategy"]
        site_row = site_screen.loc[
            site_screen["model"].eq(row["model"]) &
            site_screen["selection_strategy"].eq(strategy) &
            site_screen["cohort"].eq("FRDA")
        ]
        control_site_row = site_screen.loc[
            site_screen["model"].eq(row["model"]) &
            site_screen["selection_strategy"].eq(strategy) &
            site_screen["cohort"].eq("Control")
        ]
        rows.append({
            "model_family": model_name,
            "model": row["model"],
            "selection_strategy": strategy,
            "frda_pooled_d_z": row.get("frda_pooled_annual_d_z", np.nan),
            "control_pooled_d_z": row.get("control_pooled_annual_d_z", np.nan),
            "signed_frda_control_contrast": row.get("signed_frda_control_contrast", np.nan),
            "frda_v1_v2_d_z": row.get("frda_v1_v2_d_z", np.nan),
            "frda_v2_v3_d_z": row.get("frda_v2_v3_d_z", np.nan),
            "frda_adjusted_site_partial_r2": site_row["site_partial_r2"].iloc[0] if not site_row.empty else np.nan,
            "frda_site_cluster_p_value": site_row["site_cluster_p_value"].iloc[0] if not site_row.empty else np.nan,
            "control_adjusted_site_partial_r2": control_site_row["site_partial_r2"].iloc[0] if not control_site_row.empty else np.nan,
            "control_site_cluster_p_value": control_site_row["site_cluster_p_value"].iloc[0] if not control_site_row.empty else np.nan,
        })
    return pd.DataFrame(rows)

acceptance = pd.concat([
    compact_acceptance("global_srm", srm_result["comparison"], srm_adjusted_site),
    compact_acceptance("patient_adaptive", adaptive_result["comparison"], adaptive_adjusted_site),
], ignore_index=True)
acceptance.to_csv(OUTPUT_DIR / "site_aware_method_decision_table.csv", index=False)
combined_recipe.to_csv(OUTPUT_DIR / "feature_recipe_comparison_three_strategies.csv", index=False)

print("Decision table: original, control-aware, and site-aware")
display(acceptance.round(4))
print("Output directory")
print(OUTPUT_DIR)

Decision table: original, control-aware, and site-aware


,model_family,model,selection_strategy,frda_pooled_d_z,control_pooled_d_z,signed_frda_control_contrast,frda_v1_v2_d_z,frda_v2_v3_d_z,frda_adjusted_site_partial_r2,frda_site_cluster_p_value,control_adjusted_site_partial_r2,control_site_cluster_p_value
0,global_srm,srm_global_linear,control_aware,0.7490,-0.0114,0.7604,0.9441,0.5787,0.0698,0.0000,0.0488,0.0002
1,global_srm,srm_global_linear,frda_only,0.7485,0.0331,0.7155,0.9593,0.5693,0.0700,0.0000,0.0528,0.0012
2,global_srm,srm_global_linear,site_aware,0.7596,-0.0106,0.7701,0.9567,0.5892,0.0693,0.0000,0.0579,0.0005
3,patient_adaptive,patient_adaptive_common_modulator,control_aware,0.6440,0.0725,0.5715,0.8092,0.4960,0.0573,0.0110,0.0236,0.0521
4,patient_adaptive,patient_adaptive_common_modulator,frda_only,0.7041,0.0805,0.6237,0.8250,0.5981,0.0740,0.0004,0.0361,0.0006
5,patient_adaptive,patient_adaptive_common_modulator,site_aware,0.6473,0.0755,0.5718,0.7836,0.5239,0.0557,0.0323,0.0337,0.0058
6,patient_adaptive,patient_adaptive_strategy_specific,control_aware,0.6617,0.0760,0.5857,0.7935,0.5375,0.0547,0.0081,0.0217,0.0713
7,patient_adaptive,patient_adaptive_strategy_specific,frda_only,0.6544,0.0759,0.5786,0.7275,0.5774,0.0611,0.0035,0.0311,0.0039
8,patient_adaptive,patient_adaptive_strategy_specific,site_aware,0.6598,0.0547,0.6050,0.7875,0.5375,0.0471,0.0180,0.0416,0.0002


Output directory
/Users/robertwang/.codex/worktrees/790f/biomarkers/results/experiments/trackfa_70_feature_comparison_v1/site_aware
